In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import tensorflow as tf
import keras
import numpy as np
from keras.models import Sequential
from keras.layers import Dropout,Flatten,Conv2D,MaxPooling2D,Dense,GlobalAveragePooling2D,BatchNormalization
from keras.regularizers import Regularizer
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.applications.vgg16 import VGG16,preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [11]:
train_data_dir='data/training_set'


In [29]:
train_data_dir='data/training_set'
validation_data_dir='data/test_set'

num_train_samples=8000
num_validation_samples=2000
batch_size=128
epochs=5

base_model=VGG16(weights='imagenet',include_top=False,input_shape=(224,224,3))

for layer in base_model.layers:
    layer.trainable=False

model=Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(256,activation='relu',kernel_initializer='he_uniform'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))

In [30]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

train_datagen=ImageDataGenerator(preprocessing_function=preprocess_input)
validation_datagen=ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator=train_datagen.flow_from_directory(
                train_data_dir,target_size=(224,224),batch_size=batch_size,class_mode='binary')    
validation_generator=validation_datagen.flow_from_directory(
                validation_data_dir,target_size=(224,224),batch_size=batch_size,class_mode='binary')

history=model.fit(train_generator,
          steps_per_epoch=num_train_samples//batch_size,
          epochs=3,
          validation_data=validation_generator,
          validation_steps=num_validation_samples//batch_size)

model.save('vgg16_finetuned.h5')

Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.
Epoch 1/3
62/62 ━━━━━━━━━━━━━━━━━━━━ 1447s 23s/step - accuracy: 0.9486 - loss: 1.8206 - val_accuracy: 0.9755 - val_loss: 0.2964
Epoch 2/3
 1/62 ━━━━━━━━━━━━━━━━━━━━ 17:24 17s/step - accuracy: 0.9609 - loss: 0.1275

e:\Deep Learning\venv\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


62/62 ━━━━━━━━━━━━━━━━━━━━ 289s 4s/step - accuracy: 0.9609 - loss: 0.1275 - val_accuracy: 0.9750 - val_loss: 0.2855
Epoch 3/3
62/62 ━━━━━━━━━━━━━━━━━━━━ 1324s 21s/step - accuracy: 0.9864 - loss: 0.0726 - val_accuracy: 0.9807 - val_loss: 0.1819


In [20]:
from tensorflow.keras.models import load_model

model = load_model("vgg16_finetuned.h5")
print("Model loaded successfully")

Model loaded successfully


In [21]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input

img_path = "test_images/dog_image.jpg"  # put any image here

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

prediction = model.predict(img_array)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step
[[1.]]


In [22]:
if prediction[0][0] > 0.5:
    print("Dog")
else:
    print("Cat")

Dog


In [23]:
loss, accuracy = model.evaluate(validation_generator)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

64/64 ━━━━━━━━━━━━━━━━━━━━ 316s 5s/step - accuracy: 0.9723 - loss: 0.2503
Validation Loss: 0.25026100873947144
Validation Accuracy: 0.9723183512687683


In [24]:
class_labels = train_generator.class_indices
print(class_labels)



{'cats': 0, 'dogs': 1}


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.show()
